# GroceryCo Audit - 01 - Database Architecture

**Task 1, Part A.** Design a normalized relational database (1NF -> 2NF -> 3NF) for the GroceryCo platform, load all 5 CSVs into SQLite with foreign-key constraints, and verify referential integrity.

**Schema entities:** `departments` (21) -> `aisles` (134) -> `products` (49,688) -> `order_items` (1.38M) <- `orders` (131K train).

**Why this matters for the audit:** the normalized schema lets us compute department- and aisle-level reorder statistics with clean joins, which is the foundation for the SPC control charts in Notebook 03 and the reorder-cascade analysis in Notebook 02.

## Setup - auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 01/
#     ├── Track 01/                                (this audit)
#     │   ├── dataset/                             (5 CSVs from Instacart Kaggle)
#     │   └── grocery_audit/
#     │       ├── data/
#     │       ├── outputs/
#     │       ├── figures/
#     │       └── (notebooks live here)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "grocery_audit":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate the 5 CSVs - accept dataset/ or data/
EXPECTED = ["aisles.csv", "departments.csv", "products.csv", "orders.csv", "order_products__train.csv"]
csv_paths = {}
for name in EXPECTED:
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[name] = folder / name
            break
missing = [f for f in EXPECTED if f not in csv_paths]
assert not missing, (
    f"Missing CSVs: {missing}. Place them in {DATASET_DIR} or {DATA_DIR}"
)
print(f"Found {len(csv_paths)} CSVs: {list(csv_paths.keys())}")


## Step 1 - Load all 5 CSVs and inspect

In [ ]:
import pandas as pd
import numpy as np

aisles = pd.read_csv(csv_paths["aisles.csv"])
depts  = pd.read_csv(csv_paths["departments.csv"])
prods  = pd.read_csv(csv_paths["products.csv"])
orders = pd.read_csv(csv_paths["orders.csv"])
opt    = pd.read_csv(csv_paths["order_products__train.csv"])

print(f"aisles       : {len(aisles):>10,} rows, cols={list(aisles.columns)}")
print(f"departments  : {len(depts):>10,} rows, cols={list(depts.columns)}")
print(f"products     : {len(prods):>10,} rows, cols={list(prods.columns)}")
print(f"orders       : {len(orders):>10,} rows, cols={list(orders.columns)}")
print(f"order_items  : {len(opt):>10,} rows, cols={list(opt.columns)}")


## Step 2 - Filter to TRAIN orders only

The `orders.csv` file contains 3 sets: prior, train, test. The `order_products__train.csv` file only has line items for the train set. We filter `orders` to just train orders to match. This is the in-scope universe for the audit (131K orders, 1.38M line items).

In [ ]:
train_orders = orders[orders["eval_set"] == "train"].copy()
print(f"Train orders: {len(train_orders):,} (filtered from {len(orders):,})")
print(f"Line items:   {len(opt):,}")

# Verify referential integrity BEFORE normalising
order_ids_in_items = set(opt["order_id"].unique())
order_ids_in_train = set(train_orders["order_id"].unique())
overlap = order_ids_in_items & order_ids_in_train
print(f"\nReferential check:")
print(f"  unique order_ids in line items : {len(order_ids_in_items):,}")
print(f"  unique order_ids in train      : {len(order_ids_in_train):,}")
print(f"  overlap                        : {len(overlap):,}")
print(f"  in items but not in train      : {len(order_ids_in_items - order_ids_in_train):,}")
print(f"  in train but not in items      : {len(order_ids_in_train - order_ids_in_items):,}")


## Step 3 - Normalization walkthrough (1NF -> 2NF -> 3NF)

**1NF (atomic values)**: Every cell holds a single value, every row uniquely identifiable. All 5 source CSVs already satisfy 1NF - product_name is a single string, no repeating groups, primary keys exist (`product_id`, `order_id`, etc.).

**2NF (no partial dependencies on composite keys)**: The only table with a composite key is `order_items` (`order_id`, `product_id`). Its non-key attributes (`add_to_cart_order`, `reordered`) depend on BOTH keys together, not just one - so 2NF holds.

**3NF (no transitive dependencies)**: In `products`, the original schema has `product_id -> aisle_id -> aisle_name` (transitive). 3NF requires we move `aisle_name` into its own `aisles` table - which the source data already does. Same for `department_id -> department_name`. So the source schema IS already in 3NF.

**Conclusion:** the Instacart schema is well-designed and already 3NF-compliant. Our job is to load it into SQLite with explicit foreign-key constraints to enforce that structure at the database level.

### ER Diagram (text representation)

In [ ]:
er_diagram = '''
+-------------------+        +-------------------+        +-------------------+
|   departments     |        |     aisles        |        |     products      |
+-------------------+        +-------------------+        +-------------------+
| PK department_id  |<-+     | PK aisle_id       |<-+  +->| PK product_id     |
|    department     |  |     |    aisle          |  |  |  |    product_name   |
+-------------------+  |     +-------------------+  |  |  | FK aisle_id       |---+
                       |                            |  |  | FK department_id  |--+|
                       +----------------------------+--|--+                       ||
                                                    |  +---------------------------+
                                                    |                              |
                                                    |  +-------------------+       |
                                                    |  |   order_items     |       |
                                                    |  +-------------------+       |
                                                    |  | PK,FK order_id    |--+    |
                                                    |  | PK,FK product_id  |--|----+
                                                    |  |    add_to_cart    |  |
                                                    |  |    reordered      |  |
                                                    |  +-------------------+  |
                                                    |                         |
                                                    |  +-------------------+  |
                                                    |  |     orders        |  |
                                                    |  +-------------------+  |
                                                    |  | PK order_id       |<-+
                                                    |  |    user_id        |
                                                    |  |    eval_set       |
                                                    |  |    order_number   |
                                                    |  |    order_dow      |
                                                    |  |    order_hour     |
                                                    |  |    days_since...  |
                                                    |  +-------------------+
'''
print(er_diagram)


## Step 4 - Build SQLite database with FK constraints

In [ ]:
import sqlite3

db_path = OUTPUTS_DIR / "groceryco.db"
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

# DDL with explicit FK constraints
ddl = '''
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department    TEXT NOT NULL
);

CREATE TABLE aisles (
    aisle_id INTEGER PRIMARY KEY,
    aisle    TEXT NOT NULL
);

CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    product_name  TEXT NOT NULL,
    aisle_id      INTEGER NOT NULL,
    department_id INTEGER NOT NULL,
    FOREIGN KEY (aisle_id)      REFERENCES aisles(aisle_id),
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE orders (
    order_id              INTEGER PRIMARY KEY,
    user_id               INTEGER NOT NULL,
    eval_set              TEXT NOT NULL,
    order_number          INTEGER NOT NULL,
    order_dow             INTEGER NOT NULL,
    order_hour_of_day     INTEGER NOT NULL,
    days_since_prior_order REAL
);

CREATE TABLE order_items (
    order_id          INTEGER NOT NULL,
    product_id        INTEGER NOT NULL,
    add_to_cart_order INTEGER NOT NULL,
    reordered         INTEGER NOT NULL CHECK (reordered IN (0,1)),
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE INDEX idx_oi_order_id   ON order_items(order_id);
CREATE INDEX idx_oi_product_id ON order_items(product_id);
CREATE INDEX idx_p_dept        ON products(department_id);
CREATE INDEX idx_p_aisle       ON products(aisle_id);
CREATE INDEX idx_o_dow         ON orders(order_dow);
CREATE INDEX idx_o_hour        ON orders(order_hour_of_day);
'''
for stmt in ddl.split(";"):
    if stmt.strip():
        cur.execute(stmt)
conn.commit()
print("Schema created with foreign-key constraints and indexes")


## Step 5 - Load data with bulk inserts

In [ ]:
# Load lookup tables first (departments, aisles), then products, then orders, finally order_items
depts.to_sql("departments", conn, if_exists="append", index=False)
aisles.to_sql("aisles", conn, if_exists="append", index=False)
prods.to_sql("products", conn, if_exists="append", index=False)
train_orders.to_sql("orders", conn, if_exists="append", index=False)
opt.to_sql("order_items", conn, if_exists="append", index=False, chunksize=50000)
conn.commit()

# Verify counts
for tbl in ["departments", "aisles", "products", "orders", "order_items"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15} : {n:>10,} rows")


## Step 6 - Verify referential integrity

In [ ]:
# Check for FK violations
print("Referential integrity check:")
checks = [
    ("products.aisle_id -> aisles",
     "SELECT COUNT(*) FROM products p LEFT JOIN aisles a ON p.aisle_id=a.aisle_id WHERE a.aisle_id IS NULL"),
    ("products.department_id -> departments",
     "SELECT COUNT(*) FROM products p LEFT JOIN departments d ON p.department_id=d.department_id WHERE d.department_id IS NULL"),
    ("order_items.order_id -> orders",
     "SELECT COUNT(*) FROM order_items oi LEFT JOIN orders o ON oi.order_id=o.order_id WHERE o.order_id IS NULL"),
    ("order_items.product_id -> products",
     "SELECT COUNT(*) FROM order_items oi LEFT JOIN products p ON oi.product_id=p.product_id WHERE p.product_id IS NULL"),
]
for label, sql in checks:
    n = cur.execute(sql).fetchone()[0]
    status = "PASS" if n == 0 else "FAIL"
    print(f"  [{status}] {label}: {n} orphaned rows")

# A reasonable smoke-test query
print("\n=== Top 5 departments by line-item volume ===")
sql = '''
SELECT d.department, COUNT(*) AS n_items
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY n_items DESC
LIMIT 5
'''
for row in cur.execute(sql):
    print(f"  {row[0]:<20} {row[1]:>10,}")

conn.close()
print(f"\nDatabase saved: {db_path}")
print(f"Size: {db_path.stat().st_size / 1024 / 1024:.1f} MB")


## Step 7 - Generate ER diagram as PNG (for the final PDF)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14); ax.set_ylim(0, 10)
ax.axis("off")

NAVY = "#1F3864"; TEAL = "#2E7D7D"; LIGHT = "#E8EEF5"

def entity_box(ax, x, y, w, h, title, fields):
    # Title bar
    title_box = FancyBboxPatch((x, y+h-0.6), w, 0.6,
                                boxstyle="round,pad=0.02",
                                facecolor=NAVY, edgecolor="black", linewidth=1.2)
    ax.add_patch(title_box)
    ax.text(x+w/2, y+h-0.3, title, ha="center", va="center",
            fontsize=12, fontweight="bold", color="white")
    # Body
    body = patches.Rectangle((x, y), w, h-0.6, facecolor=LIGHT, edgecolor="black", linewidth=1.2)
    ax.add_patch(body)
    for i, f in enumerate(fields):
        ax.text(x+0.1, y+h-0.85-i*0.32, f, ha="left", va="top",
                fontsize=9, family="monospace")

# Layout entities
entity_box(ax, 0.5, 6.5, 2.5, 2.5, "departments",
           ["PK department_id", "   department"])
entity_box(ax, 0.5, 2.5, 2.5, 2.8, "aisles",
           ["PK aisle_id", "   aisle"])
entity_box(ax, 4.5, 4.5, 3.2, 3.5, "products",
           ["PK product_id", "   product_name", "FK aisle_id", "FK department_id"])
entity_box(ax, 9.0, 4.5, 3.5, 3.8, "orders",
           ["PK order_id", "   user_id", "   eval_set", "   order_number",
            "   order_dow", "   order_hour_of_day", "   days_since_prior_order"])
entity_box(ax, 5.0, 0.5, 4.5, 2.8, "order_items",
           ["PK,FK order_id", "PK,FK product_id", "   add_to_cart_order",
            "   reordered (0/1)"])

# Relationships (arrows)
def rel(ax, x1, y1, x2, y2, label="", offset=0):
    arr = FancyArrowPatch((x1, y1), (x2, y2),
                          arrowstyle="-|>", mutation_scale=18,
                          color=TEAL, linewidth=1.5)
    ax.add_patch(arr)
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2 + offset, label,
                ha="center", fontsize=8, style="italic", color=TEAL,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none"))

rel(ax, 4.5, 7.0, 3.0, 7.5, "1:N", offset=0.15)        # depts -> products
rel(ax, 4.5, 5.5, 3.0, 4.0, "1:N", offset=0.15)        # aisles -> products
rel(ax, 6.1, 4.5, 6.5, 3.3, "1:N", offset=0.1)         # products -> order_items
rel(ax, 9.0, 5.5, 8.5, 3.3, "1:N", offset=0.1)         # orders -> order_items

ax.set_title("GroceryCo - 3NF Entity-Relationship Diagram",
             fontsize=15, fontweight="bold", pad=15, color=NAVY)
ax.text(7, 9.7, "5 entities, 4 foreign-key relationships, 3NF compliant",
        ha="center", fontsize=10, style="italic", color="#595959")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "er_diagram.png", dpi=160, bbox_inches="tight",
            facecolor="white")
plt.show()
print(f"Saved: {FIGURES_DIR / 'er_diagram.png'}")


**Notebook 01 complete.** SQLite database is at `outputs/groceryco.db` with 5 tables, FK constraints, indexes, and zero orphaned rows. Move on to `02_complex_joins_cascade.ipynb`.